# 03 — Análisis Exploratorio (EDA)

**Fase 3 del proyecto ISO 20022 en Redes DLT.**

Toma el dataset etiquetado de `data/processed/txs_etiquetadas.csv` y produce gráficos de series temporales, distribuciones, frecuencia de códigos ISO, concentración de liquidez y correlaciones, junto con CSVs de estadísticas descriptivas. Los resultados se guardan en `outputs/reportes/`.

> **Nota:** los montos de Stellar llegan vacíos en v1 (limitación heredada de la Fase 2), por lo que esa red aporta a los conteos pero no a los análisis basados en `amount`.

# 1. Configuración

In [ ]:
DEBUG: bool = True       # activa prints de verificación
SAVE_DATA: bool = False  # activa guardado de figuras y CSV

# 2. Imports

In [ ]:
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")

# 3. Paths y Constantes

In [ ]:
# Paths de entrada / salida
BASE_DIR: str = "tesis"
ARCHIVO_ENTRADA: str = os.path.join(
    BASE_DIR, "data", "processed", "txs_etiquetadas.csv")
DIR_REPORTES: str = os.path.join(
    BASE_DIR, "outputs", "reportes")
DIR_FIGURAS: str = os.path.join(DIR_REPORTES, "figuras_eda")

# Constantes de análisis
GRANULARIDAD: str = "1D"  # agregación diaria
REDES: list[str] = ["xrpl", "stellar", "hedera"]
ORDEN_ISO: list[str] = ["LQDT", "COLL", "TREA", "INTC", "CASH"]
COLUMNAS: list[str] = [
    "tx_id", "red", "timestamp", "emisor", "receptor",
    "amount", "currency", "fee", "memo_raw",
    "memo_decoded", "iso_code", "iso_metodo",
]

# 4. Funciones

### 4.a — Carga y guardado

In [ ]:
def cargar_dataset(ruta: str) -> pd.DataFrame:
    """Carga el dataset etiquetado y parsea el timestamp."""
    if not os.path.exists(ruta):
        if DEBUG:
            print(f"[DEBUG] no existe {ruta}")
        return pd.DataFrame(columns=COLUMNAS)
    df = pd.read_csv(ruta)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(
            df["timestamp"], utc=True, errors="coerce")
    if DEBUG:
        print(f"[DEBUG] filas cargadas: {len(df)}")
    return df


def guardar_csv(df: pd.DataFrame, nombre: str) -> None:
    """Guarda un DataFrame en outputs/reportes/ si procede."""
    if not SAVE_DATA:
        return
    os.makedirs(DIR_REPORTES, exist_ok=True)
    ruta = os.path.join(DIR_REPORTES, nombre)
    df.to_csv(ruta, index=False)
    if DEBUG:
        print(f"[DEBUG] guardado: {ruta}")


def guardar_figura(fig, nombre: str) -> None:
    """Guarda una figura matplotlib como PNG si procede."""
    if not SAVE_DATA:
        return
    os.makedirs(DIR_FIGURAS, exist_ok=True)
    ruta = os.path.join(DIR_FIGURAS, nombre)
    fig.savefig(ruta, dpi=120, bbox_inches="tight")
    if DEBUG:
        print(f"[DEBUG] figura: {ruta}")


def guardar_figura_plotly(fig, nombre: str) -> None:
    """Guarda una figura plotly como HTML si procede."""
    if not SAVE_DATA:
        return
    os.makedirs(DIR_FIGURAS, exist_ok=True)
    ruta = os.path.join(DIR_FIGURAS, nombre)
    fig.write_html(ruta)
    if DEBUG:
        print(f"[DEBUG] figura: {ruta}")

### 4.b — Estadísticas descriptivas

In [ ]:
def estadisticas_descriptivas(df: pd.DataFrame) -> pd.DataFrame:
    """Estadísticas de amount y fee por red y código ISO."""
    if df.empty:
        return pd.DataFrame()
    stats = (
        df.groupby(["red", "iso_code"], dropna=False)
        .agg(
            n_txs=("tx_id", "count"),
            monto_total=("amount", "sum"),
            monto_medio=("amount", "mean"),
            monto_mediano=("amount", "median"),
            monto_std=("amount", "std"),
            fee_medio=("fee", "mean"),
        )
        .reset_index()
    )
    return stats

### 4.c — Series temporales, distribuciones y frecuencias

In [ ]:
def serie_temporal_volumen(df: pd.DataFrame):
    """Serie temporal (diaria) de nº de txs por red."""
    if df.empty:
        return None
    base = df.dropna(subset=["timestamp"]).copy()
    if base.empty:
        return None
    base["fecha"] = base["timestamp"].dt.floor("D")
    serie = (
        base.groupby(["fecha", "red"])
        .size()
        .reset_index(name="n_txs")
    )
    fig = px.line(
        serie, x="fecha", y="n_txs", color="red",
        title="Volumen diario de transacciones por red")
    return fig


def distribucion_montos(df: pd.DataFrame):
    """Boxplot de montos por red en escala logarítmica."""
    datos = df.dropna(subset=["amount"])
    datos = datos[datos["amount"] > 0]
    if datos.empty:
        return None
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=datos, x="red", y="amount", ax=ax)
    ax.set_yscale("log")
    ax.set_title("Distribución de montos por red (log)")
    ax.set_ylabel("amount (escala log)")
    fig.tight_layout()
    return fig


def frecuencia_iso(df: pd.DataFrame):
    """Frecuencia de códigos ISO 20022 por red."""
    if df.empty:
        return None
    datos = df.copy()
    datos["iso_code"] = datos["iso_code"].fillna("sin_clasif")
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.countplot(
        data=datos, x="iso_code", hue="red",
        order=ORDEN_ISO + ["sin_clasif"], ax=ax)
    ax.set_title("Frecuencia de códigos ISO 20022 por red")
    fig.tight_layout()
    return fig

### 4.d — Concentración de liquidez y correlaciones

In [ ]:
def concentracion_liquidez(df: pd.DataFrame,
                           top: int = 10) -> pd.DataFrame:
    """Cuota de volumen de los top emisores por red."""
    datos = df.dropna(subset=["amount"])
    datos = datos[datos["amount"] > 0]
    filas = []
    for red, grupo in datos.groupby("red"):
        total = grupo["amount"].sum()
        if total == 0:
            continue
        top_vol = (
            grupo.groupby("emisor")["amount"].sum()
            .sort_values(ascending=False)
            .head(top).sum())
        filas.append({
            "red": red,
            "emisores_unicos": grupo["emisor"].nunique(),
            "cuota_top": top_vol / total,
        })
    return pd.DataFrame(filas)


def correlaciones_diarias(df: pd.DataFrame):
    """Heatmap de correlación entre métricas diarias."""
    base = df.dropna(subset=["timestamp"]).copy()
    if base.empty:
        return None, pd.DataFrame()
    base["fecha"] = base["timestamp"].dt.floor("D")
    diario = base.groupby("fecha").agg(
        n_txs=("tx_id", "count"),
        monto_total=("amount", "sum"),
        fee_total=("fee", "sum"),
        monto_medio=("amount", "mean"),
    )
    corr = diario.corr()
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        corr, annot=True, cmap="coolwarm",
        vmin=-1, vmax=1, ax=ax)
    ax.set_title("Correlación entre métricas diarias")
    fig.tight_layout()
    return fig, corr.reset_index()

# 5. Proceso

In [ ]:
df = cargar_dataset(ARCHIVO_ENTRADA)

# 1. Estadísticas descriptivas por red y código ISO
stats = estadisticas_descriptivas(df)
guardar_csv(stats, "eda_estadisticas_descriptivas.csv")
if DEBUG and not stats.empty:
    print(stats.head())

# 2. Serie temporal de volumen (interactiva, plotly)
fig_vol = serie_temporal_volumen(df)
if fig_vol is not None:
    guardar_figura_plotly(fig_vol, "eda_volumen_diario.html")
    fig_vol.show()

# 3. Distribución de montos por red
fig_montos = distribucion_montos(df)
if fig_montos is not None:
    guardar_figura(fig_montos, "eda_distribucion_montos.png")

# 4. Frecuencia de códigos ISO 20022
fig_iso = frecuencia_iso(df)
if fig_iso is not None:
    guardar_figura(fig_iso, "eda_frecuencia_iso.png")

# 5. Concentración de liquidez
conc = concentracion_liquidez(df)
guardar_csv(conc, "eda_concentracion_liquidez.csv")
if DEBUG and not conc.empty:
    print(conc)

# 6. Correlaciones entre métricas diarias
fig_corr, corr = correlaciones_diarias(df)
if fig_corr is not None:
    guardar_figura(fig_corr, "eda_correlaciones.png")
    guardar_csv(corr, "eda_correlaciones.csv")

if DEBUG:
    print("[DEBUG] EDA finalizado")